In [ ]:
from ultralytics import YOLO
import cv2
import csv
import datetime

# CONFIG 
model_path = r"D:\orignals _All_project\Licence Number Plate_Detection\best.pt" 
url = "http://192.168.0.193:8080/video"

# Load Model
model = YOLO(model_path)

# Camera Stream
cap = cv2.VideoCapture(url)
if not cap.isOpened():
    print("Error: Mobile camera stream open nahi ho raha")
    exit()

frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30

# Video Writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("number_plate_output.mp4", fourcc, fps, (frame_width, frame_height))

# CSV File for logging
csv_file = open("detected_plates.csv", mode="w", newline="")
csv_writer = csv.writer(csv_file)
csv_writer.writerow(["Timestamp", "Plate_Number"])   # headers

print(" System started. Press Q to stop.")

# Loop
while True:
    ret, frame = cap.read()
    if not ret:
        print("Frame read error - check connection")
        break

    # YOLO Detection
    results = model.predict(frame, conf=0.5)

    annotated_frame = results[0].plot()  # bounding boxes draw

    # Extract detected plates
    for box in results[0].boxes:
        cls_id = int(box.cls[0])  # class ID
        label = model.names[cls_id]  # class name (e.g. "plate")

        # Crop detected plate region
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        plate_img = frame[y1:y2, x1:x2]

        # OCR (recognition of plate number)
        #  For OCR we can use EasyOCR
        import easyocr
        reader = easyocr.Reader(['en'])
        try:
            ocr_result = reader.readtext(plate_img)
            if len(ocr_result) > 0:
                plate_text = ocr_result[0][-2]  # detected text
                timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                # Save to CSV
                csv_writer.writerow([timestamp, plate_text])
                print(f"Detected Plate: {plate_text}")
        except:
            pass

    # Show live window
    cv2.imshow("YOLO Licence Plate Detection", annotated_frame)

    # Save video
    out.write(annotated_frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
csv_file.close()
cv2.destroyAllWindows()
print("Detection finished. Video saved as number_plate_output.mp4, CSV saved as detected_plates.csv")

 System started. Press Q to stop.

0: 384x640 (no detections), 229.6ms
Speed: 11.9ms preprocess, 229.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 188.3ms
Speed: 10.5ms preprocess, 188.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 216.4ms
Speed: 4.7ms preprocess, 216.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 158.8ms
Speed: 4.9ms preprocess, 158.8ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 151.0ms
Speed: 21.6ms preprocess, 151.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 152.2ms
Speed: 4.9ms preprocess, 152.2ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 149.1ms
Speed: 5.7ms preprocess, 149.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no de

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
